# Language Debug Notebook (I18N)

This notebook is for **diagnosing non-English scoring behavior** end-to-end.

It will:
1. Pull up to 50 jobs from **We Work Remotely RSS**
2. Detect language for each job post
3. Attempt translation to English (when confident non-English)
4. Display **all fields** in original and translated versions
5. Surface translation errors for quick triage


In [5]:
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

if importlib.util.find_spec('deep_translator') is None:
    print('Installing deep-translator in the active notebook kernel...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'deep-translator==1.11.4'])
else:
    print('deep-translator is already available in this kernel.')

import job_fraud_detector.i18n as i18n_module
importlib.reload(i18n_module)
normalize_posting_language = i18n_module.normalize_posting_language
TRANSLATABLE_FIELDS = i18n_module.TRANSLATABLE_FIELDS

from job_fraud_detector.live_sources import fetch_jobs_from_sources

print('i18n module loaded from:', i18n_module.__file__)

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.width', 240)

ROOT



deep-translator is already available in this kernel.
i18n module loaded from: /Users/shahan/Documents/Remote Job Scam-NotScam/src/job_fraud_detector/i18n.py


PosixPath('/Users/shahan/Documents/Remote Job Scam-NotScam')

In [18]:
import sys, importlib.util
import job_fraud_detector.i18n as i18n_module
print(sys.executable)
print(i18n_module.__file__)
print("deep_translator:", bool(importlib.util.find_spec("deep_translator")))

/Users/shahan/Documents/Remote Job Scam-NotScam/.venv/bin/python
/Users/shahan/Documents/Remote Job Scam-NotScam/src/job_fraud_detector/i18n.py
deep_translator: True


In [20]:
SOURCES = {
    'we_work_remotely_rss': 'https://weworkremotely.com/remote-jobs.rss',
}
PER_SOURCE = 50
ENABLE_TRANSLATION = True
SHOW_FULL_TEXT = True
FIELDS_TO_COMPARE = [
    'title',
    'company_profile',
    'location',
    'department',
    'description',
    'requirements',
    'benefits',
    'employment_type',
    'required_experience',
    'required_education',
    'industry',
    'function',
    'salary_range',
]

print('Configured source:', SOURCES)
print('PER_SOURCE =', PER_SOURCE)
print('ENABLE_TRANSLATION =', ENABLE_TRANSLATION)

Configured source: {'we_work_remotely_rss': 'https://weworkremotely.com/remote-jobs.rss'}
PER_SOURCE = 50
ENABLE_TRANSLATION = True


In [9]:
jobs = fetch_jobs_from_sources(
    sources=SOURCES,
    per_source=PER_SOURCE,
    fail_fast=False,
)

print(f'Fetched {len(jobs)} jobs from We Work Remotely RSS.')

if not jobs:
    print('No jobs fetched. Check network and rerun.')

Fetched 50 jobs from We Work Remotely RSS.


In [21]:
language_rows = []
detailed_posts = []

# Safety: reload i18n each run so notebook does not hold stale function objects.
import importlib
import job_fraud_detector.i18n as i18n_module
importlib.reload(i18n_module)
normalize_posting_language = i18n_module.normalize_posting_language

for idx, job in enumerate(jobs, start=1):
    i18n = normalize_posting_language(job, enable_translation=ENABLE_TRANSLATION)
    translated_posting = i18n.posting

    language_rows.append({
        'row_id': idx,
        'source': job.get('source', ''),
        'title': job.get('title', ''),
        'detected_language': i18n.language,
        'language_confidence': round(i18n.language_confidence, 4),
        'language_detector': i18n.language_detector,
        'translation_applied': i18n.translation_applied,
        'translation_provider': i18n.translation_provider,
        'translation_error': i18n.translation_error,
        'job_url': job.get('job_url', ''),
        'apply_url': job.get('apply_url', ''),
    })

    original = {field: str(job.get(field, '') or '') for field in FIELDS_TO_COMPARE}
    translated = {field: str(translated_posting.get(field, '') or '') for field in FIELDS_TO_COMPARE}

    detailed_posts.append({
        'row_id': idx,
        'source': job.get('source', ''),
        'job_url': job.get('job_url', ''),
        'apply_url': job.get('apply_url', ''),
        'detected_language': i18n.language,
        'language_confidence': round(i18n.language_confidence, 4),
        'language_detector': i18n.language_detector,
        'translation_applied': i18n.translation_applied,
        'translation_provider': i18n.translation_provider,
        'translation_error': i18n.translation_error,
        'original': original,
        'translated': translated,
    })


debug_summary_df = pd.DataFrame(language_rows)
debug_detail_df = pd.DataFrame(detailed_posts)

print('Rows processed:', len(debug_summary_df))



Rows processed: 50


In [22]:
display(debug_summary_df)

display(
    debug_summary_df
    .groupby(['detected_language', 'translation_applied'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

translation_failures = debug_summary_df[debug_summary_df['translation_error'].notna()]
if not translation_failures.empty:
    print('Translation failures:')
    display(translation_failures[['row_id', 'title', 'detected_language', 'translation_error', 'job_url']])
else:
    print('No translation failures detected.')

,row_id,source,title,detected_language,language_confidence,language_detector,translation_applied,translation_provider,translation_error,job_url,apply_url
0,1,we_work_remotely_rss,"Senior Software Engineer, Platform",en,0.55,latin-keyword-heuristic,False,none,None,https://weworkremotely.com/remote-jobs/speechify-inc-senior-software-engineer-platform-1,https://weworkremotely.com/remote-jobs/speechify-inc-senior-software-engineer-platform-1
1,2,we_work_remotely_rss,"Senior Software Engineer, Web",en,0.55,latin-keyword-heuristic,False,none,None,https://weworkremotely.com/remote-jobs/speechify-inc-senior-software-engineer-web-1,https://weworkremotely.com/remote-jobs/speechify-inc-senior-software-engineer-web-1
2,3,we_work_remotely_rss,Junior Full-Stack Developer (Laravel and VueJS),en,0.55,latin-keyword-heuristic,False,none,None,https://weworkremotely.com/remote-jobs/appetiser-junior-full-stack-developer-laravel-and-vuejs,https://weworkremotely.com/remote-jobs/appetiser-junior-full-stack-developer-laravel-and-vuejs
3,4,we_work_remotely_rss,Product Marketing Manager (B2B for life sciences/pharma/biotech),en,0.55,latin-keyword-heuristic,False,none,None,https://weworkremotely.com/remote-jobs/viseven-product-marketing-manager-b2b-for-life-sciences-pharma-biotech,https://weworkremotely.com/remote-jobs/viseven-product-marketing-manager-b2b-for-life-sciences-pharma-biotech
4,5,we_work_remotely_rss,"Director, AI Product Development",en,0.55,latin-keyword-heuristic,False,none,None,https://weworkremotely.com/remote-jobs/centralreach-director-ai-product-development,https://weworkremotely.com/remote-jobs/centralreach-director-ai-product-development
5,6,we_work_remotely_rss,Senior DevOps Engineer - Migraciones CI/CD (Remoto 100%),es,0.95,latin-keyword-heuristic,True,deep-translator-google,None,https://weworkremotely.com/remote-jobs/knowmad-mood-senior-devops-engineer-migraciones-ci-cd-remoto-100,https://weworkremotely.com/remote-jobs/knowmad-mood-senior-devops-engineer-migraciones-ci-cd-remoto-100
6,7,we_work_remotely_rss,Product Owner (m/f),it,0.95,latin-keyword-heuristic,True,deep-translator-google,None,https://weworkremotely.com/remote-jobs/iungo-spa-product-owner-m-f,https://weworkremotely.com/remote-jobs/iungo-spa-product-owner-m-f
7,8,we_work_remotely_rss,Product Owner Zendesk Customer Service (m/w/d) // remote möglich,de,0.85,latin-keyword-heuristic,True,deep-translator-google,None,https://weworkremotely.com/remote-jobs/e-breuninger-co-product-owner-zendesk-customer-service-m-w-d-remote-moglich,https://weworkremotely.com/remote-jobs/e-breuninger-co-product-owner-zendesk-customer-service-m-w-d-remote-moglich
8,9,we_work_remotely_rss,"Senior Product Analyst, Remote",en,0.55,latin-keyword-heuristic,False,none,None,https://weworkremotely.com/remote-jobs/aledade-senior-product-analyst-remote,https://weworkremotely.com/remote-jobs/aledade-senior-product-analyst-remote
9,10,we_work_remotely_rss,Product Manager,en,0.55,latin-keyword-heuristic,False,none,None,https://weworkremotely.com/remote-jobs/welocalize-product-manager,https://weworkremotely.com/remote-jobs/welocalize-product-manager


,detected_language,translation_applied,count
1,en,False,33
4,pt,True,13
0,de,True,2
2,es,True,1
3,it,True,1


No translation failures detected.


In [23]:
def _combined_text(payload: dict[str, str]) -> str:
    parts = []
    for field in FIELDS_TO_COMPARE:
        value = str(payload.get(field, '') or '').strip()
        if value:
            parts.append(f'[{field}]\n{value}')
    return '\n\n'.join(parts)


def render_language_debug(row_id: int) -> None:
    detail = detailed_posts[row_id - 1]

    header = (
        f"### #{detail['row_id']} | {detail['original'].get('title', '')}\n"
        f"- source: {detail['source']}\n"
        f"- detected_language: {detail['detected_language']} (confidence={detail['language_confidence']})\n"
        f"- detector: {detail['language_detector']}\n"
        f"- translation_applied: {detail['translation_applied']}\n"
        f"- translation_provider: {detail['translation_provider']}\n"
        f"- translation_error: {detail['translation_error']}\n"
        f"- job_url: {detail['job_url']}\n"
        f"- apply_url: {detail['apply_url']}"
    )
    display(Markdown(header))

    compare_rows = []
    for field in FIELDS_TO_COMPARE:
        original = str(detail['original'].get(field, '') or '')
        translated = str(detail['translated'].get(field, '') or '')
        compare_rows.append({
            'field': field,
            'original': original,
            'translated': translated,
            'changed': original.strip() != translated.strip(),
        })

    compare_df = pd.DataFrame(compare_rows)
    display(compare_df)

    if SHOW_FULL_TEXT:
        original_combined = _combined_text(detail['original'])
        translated_combined = _combined_text(detail['translated'])

        display(Markdown('**Original Full Posting Text**'))
        print(original_combined if original_combined else '[empty]')

        display(Markdown('**Translated Full Posting Text**'))
        print(translated_combined if translated_combined else '[empty]')

    display(Markdown('---'))

In [24]:
# Run this cell to render all fetched jobs with full original/translated details.
START_INDEX = 1
END_INDEX = len(detailed_posts)

print(f'Rendering jobs {START_INDEX} to {END_INDEX} (inclusive).')

for row_id in range(START_INDEX, END_INDEX + 1):
    render_language_debug(row_id)

Rendering jobs 1 to 50 (inclusive).


### #1 | Senior Software Engineer, Platform
- source: we_work_remotely_rss
- detected_language: en (confidence=0.55)
- detector: latin-keyword-heuristic
- translation_applied: False
- translation_provider: none
- translation_error: None
- job_url: https://weworkremotely.com/remote-jobs/speechify-inc-senior-software-engineer-platform-1
- apply_url: https://weworkremotely.com/remote-jobs/speechify-inc-senior-software-engineer-platform-1

,field,original,translated,changed
0,title,"Senior Software Engineer, Platform","Senior Software Engineer, Platform",False
1,company_profile,Speechify Inc,Speechify Inc,False
2,location,"Anywhere in the World, Florida","Anywhere in the World, Florida",False
3,department,,,False
4,description,"Headquarters: Florida URL: http://www.speechify.com Overview As Speechify expands, our Platform team seeks a Senior Software Engineer. This role is central to ensuring our succ...","Headquarters: Florida URL: http://www.speechify.com Overview As Speechify expands, our Platform team seeks a Senior Software Engineer. This role is central to ensuring our succ...",False
5,requirements,,,False
6,benefits,,,False
7,employment_type,Full-Time,Full-Time,False
8,required_experience,,,False
9,required_education,,,False


**Original Full Posting Text**

[title]
Senior Software Engineer, Platform

[company_profile]
Speechify Inc

[location]
Anywhere in the World, Florida

[description]
Headquarters: Florida URL: http://www.speechify.com Overview As Speechify expands, our Platform team seeks a Senior Software Engineer. This role is central to ensuring our success at Speechify by working on key features like: Payments, Analytics, Subscriptions and our API. If you are passionate about strategizing, enjoy high-paced environments, and are eager to take ownership of product decisions, we’d love to hear from you. What You’ll Do Design, develop, and maintain robust APIs including Public TTS API, Internal APIs like Payment, Subscription, Auth and Consumption Tracking, ensuring they meet business and scalability requirements. Oversee the full backend API landscape, enhancing and optimizing for performance and maintainability. Collaborate on B2B solutions, focusing on customization and integration needs for enterprise clients. Work closely with c

...

**Original Full Posting Text**

[title]
Senior Product Manager

[company_profile]
Biggerpockets

[location]
Anywhere in the World

[description]
Headquarters: United States Senior Product Manager – Marketplace About Us BiggerPockets is the leading real estate investing community and podcast. Since 2004, we’ve helped millions of people take steps toward financial freedom through education, tools, and a vibrant member-driven community. With over 3 million members, BiggerPockets is the go-to platform for real estate investors at every stage of their journey. We’re building the next generation of products that power learning, investing, and community—and we’re looking for a seasoned product leader to help shape the foundation of our platform. About the Role We’re looking for a Senior Product Manager, Marketplace to own and evolve key marketplace experiences at BiggerPockets. This role will focus on building scalable, trusted, and high-performing marketplace products that create value for both sides of the network - inves

**Translated Full Posting Text**

[title]
Senior Product Manager

[company_profile]
Biggerpockets

[location]
Anywhere in the World

[description]
Headquarters: United States Senior Product Manager – Marketplace About Us BiggerPockets is the leading real estate investing community and podcast. Since 2004, we’ve helped millions of people take steps toward financial freedom through education, tools, and a vibrant member-driven community. With over 3 million members, BiggerPockets is the go-to platform for real estate investors at every stage of their journey. We’re building the next generation of products that power learning, investing, and community—and we’re looking for a seasoned product leader to help shape the foundation of our platform. About the Role We’re looking for a Senior Product Manager, Marketplace to own and evolve key marketplace experiences at BiggerPockets. This role will focus on building scalable, trusted, and high-performing marketplace products that create value for both sides of the network - inves

---

In [ ]:
# Optional quick filters for failures or non-English jobs
non_english = debug_summary_df[debug_summary_df['detected_language'].fillna('unknown') != 'en']
translation_errors = debug_summary_df[debug_summary_df['translation_error'].notna()]

print('Non-English rows:', len(non_english))
display(non_english[['row_id', 'title', 'detected_language', 'language_confidence', 'translation_applied', 'translation_error']])

print('Rows with translation errors:', len(translation_errors))
display(translation_errors[['row_id', 'title', 'detected_language', 'translation_error', 'job_url']])

Non-English rows: 17


,row_id,title,detected_language,language_confidence,translation_applied,translation_error
5,6,Senior DevOps Engineer - Migraciones CI/CD (Remoto 100%),es,0.95,True,None
6,7,Product Owner (m/f),it,0.95,True,None
7,8,Product Owner Zendesk Customer Service (m/w/d) // remote möglich,de,0.85,True,None
11,12,[Banco de Talentos] Pessoas com deficiência,pt,0.95,True,None
13,14,Desenvolvedor Dynamics 365 CE [100% Remota],pt,0.95,True,None
16,17,Senior Fullstack Engineer — Produtos Financeiros (Miniapps),pt,0.85,True,None
17,18,Pessoa Desenvolvedora Backend Sênior,pt,0.95,True,None
19,20,Profissional Web Designer Sênior,pt,0.95,True,None
20,21,Profissional Web Designer Sênior,pt,0.95,True,None
21,22,Senior Product Manager - Remote - 13.5-18.3k CLT,pt,0.95,True,None


Rows with translation errors: 0


,row_id,title,detected_language,translation_error,job_url
